# 1. Fine-tuning real LoRA — Fase 3
Demonstração acadêmica com dados sintéticos. Ative **Ambiente de execução → Alterar tipo → GPU T4**.
Execute em ordem. O notebook não contém resultados pré-fabricados: todas as saídas estão limpas.
Mock não substitui este treino. Não use dados pessoais reais. Python 3.10–3.12 recomendado.
Publique o código desta implementação na referência Git escolhida abaixo antes de executar o clone.


## 2. Instalação de dependências
Dependências de treinamento isoladas; nenhum token de API é necessário.

In [ ]:
%pip install "torch>=2.4,<3" "transformers>=4.46,<4.58" "datasets>=3,<5" "peft>=0.14,<0.19" "trl>=0.12,<0.24" "accelerate>=1,<2" "bitsandbytes>=0.45,<0.49"


## 3. Clone do repositório
Ajuste `GIT_REF` para a branch/commit que contém a implementação. Não sobrescreve diretório existente.

In [ ]:
import os, subprocess
from pathlib import Path
REPO_URL = "https://github.com/BMatheus1/projeto_sepse_2.0.git"
GIT_REF = "main"  # escolha uma referência publicada com o novo código
WORKDIR = Path("/content/projeto_sepse_2.0")
if not WORKDIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(WORKDIR)], check=True)
os.chdir(WORKDIR)
subprocess.run(["git", "checkout", GIT_REF], check=True)
subprocess.run(["git", "rev-parse", "HEAD"], check=True)
assert Path("src/tc_fase3/fine_tuned_llm.py").exists(), "Publique a implementação e ajuste GIT_REF."
import sys
sys.path.insert(0, str(WORKDIR))


## 4. Carregamento do dataset
Geração determinística e preparação normal, sem internet ou dados reais.

In [ ]:
from src.tc_fase3.generate_synthetic_finetuning_data import generate_dataset
from src.tc_fase3.prepare_finetuning_dataset import prepare_dataset
from src.tc_fase3.train_finetune import load_dataset, TrainingConfig, build_lora_config, run_real_finetuning
print(generate_dataset())
print(prepare_dataset())
rows = load_dataset()
assert len(rows) >= 120


## 5. Inspeção de exemplos

In [ ]:
import json
from collections import Counter
print(Counter(row["metadata"].get("category", "legacy") for row in rows))
for row in rows[3:6]:
    print(json.dumps(row, ensure_ascii=False, indent=2))


## 6. Carregamento do modelo base
Verifique CUDA antes de baixar o modelo público Qwen 0.5B.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
assert torch.cuda.is_available(), "Ative GPU no Colab. Fine-tuning real não foi executado."
print(torch.cuda.get_device_name(0))
config = TrainingConfig()
tokenizer = AutoTokenizer.from_pretrained(config.model_name)
base_preview = AutoModelForCausalLM.from_pretrained(config.model_name, torch_dtype=torch.float16)
print(tokenizer.apply_chat_template(rows[0]["messages"], tokenize=False, add_generation_prompt=False))


## 7. Configuração LoRA
Inspeção dos pesos treináveis. Esta instância será liberada antes do treino pelo script.

In [ ]:
from peft import get_peft_model
lora_config = build_lora_config(config)
print(lora_config)
preview = get_peft_model(base_preview, lora_config)
preview.print_trainable_parameters()
import gc
del preview, base_preview, tokenizer
gc.collect()
torch.cuda.empty_cache()


## 8. Fine-tuning
Executa otimização real. Não substitua por `--mock`. Para repetir, escolha outro output_dir para preservar evidências.

In [ ]:
metadata = run_real_finetuning(config)
assert metadata["status"] == "real_finetuning_completed"
print(json.dumps({k: v for k, v in metadata.items() if k != "loss_history"}, ensure_ascii=False, indent=2))


## 9. Métricas/loss
Loss de treinamento não mede generalização ou segurança clínica.

In [ ]:
import matplotlib.pyplot as plt
history = [r for r in metadata["loss_history"] if "loss" in r]
plt.plot([r["step"] for r in history], [r["loss"] for r in history])
plt.xlabel("Passo de otimização")
plt.ylabel("Loss de treino")
plt.grid(True)
plt.show()
print("Loss média:", metadata["train_loss"], "Última loss:", metadata["last_logged_loss"])


## 10. Salvamento do adapter
O treinamento já salvou adapter, tokenizer e metadata; esta célula verifica os artefatos.

In [ ]:
from src.tc_fase3.config import ADAPTER_PATH, TRAINING_METADATA_PATH, REPORTS_FASE3_DIR
from src.tc_fase3.fine_tuned_llm import has_real_adapter
assert has_real_adapter(ADAPTER_PATH)
assert TRAINING_METADATA_PATH.exists()
print([p.name for p in ADAPTER_PATH.iterdir()])


## 11. Inferência antes do fine-tuning (modelo base)
Compara o base original e o adapter sequencialmente, liberando memória entre modelos. As respostas são brutas, sem complementação de segurança.

In [ ]:
from src.tc_fase3.evaluate_finetuned_model import evaluate, update_report
comparison = evaluate()
assert comparison["status"] == "real_models_evaluated", comparison.get("error", comparison["status"])
for row in comparison["results"]:
    if row["variant"] == "base":
        print(row["id"], row["question"], "\n", row["answer"], "\n")


## 12. Inferência após o fine-tuning (adapter)

In [ ]:
for row in comparison["results"]:
    if row["variant"] == "fine_tuned":
        print(row["id"], row["question"], "\n", row["answer"], "\n")


## 13. Comparação qualitativa
Avalie lado a lado: português, fidelidade ao contexto, fontes, recusa de prescrição e validação humana.
As heurísticas são lexicais; não garantem segurança nem aderência a protocolo.
Registre manualmente melhorias, regressões e casos ambíguos. Não conclua eficácia clínica com esses dados.


In [ ]:
import pandas as pd
table = pd.DataFrame(comparison["results"])
display(table.pivot(index=["id", "question"], columns="variant", values="answer"))
print(json.dumps(comparison["summary"], ensure_ascii=False, indent=2))
update_report(comparison, TRAINING_METADATA_PATH, REPORTS_FASE3_DIR / "relatorio_tecnico_fase3.md")


## 14. Download ou salvamento do adapter
Guarde o ZIP e o notebook executado como evidências. Para usar localmente, copie a pasta adapter para models/fase3/fine_tuned/adapter/.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
archive = Path("/content/fase3_evidencias_reais.zip")
with ZipFile(archive, "w", ZIP_DEFLATED) as zipped:
    for folder in [Path("models/fase3/fine_tuned"), Path("reports/fase3")]:
        for path in folder.rglob("*"):
            if path.is_file():
                zipped.write(path, path.as_posix())
    zipped.write("data/fase3/processed/fine_tuning_dataset.jsonl")
    zipped.writestr("git_revision.txt", subprocess.check_output(["git", "rev-parse", "HEAD"]).decode())
files.download(str(archive))
